In [6]:
import torch
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation
import numpy as np
import cv2
import warnings
warnings.filterwarnings("ignore", category=FutureWarning) # huggingface is mad

In [ ]:
input_video = 'data/test_2.mp4'
output_video = 'data/new_output_2.mp4'
TARGET_SIZE = 1024
SHIRT_LABEL = 4
BATCH_SIZE = 8

ALPHA = 0.35                  # mask transparency
MASK_COLOR = (0, 255, 0)      # (Blue, Green, Red)

In [3]:
model_name = "mattmdjaga/segformer_b2_clothes"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
processor = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
model = AutoModelForSemanticSegmentation.from_pretrained("mattmdjaga/segformer_b2_clothes").to(device) # move to gpu if available

model.eval()

print(f'Inference running on {device}')


Inference running on cuda


In [4]:
from video_segmentation import get_single_mask_from_video

shirt_tensor = get_single_mask_from_video(input_video, target_resolution=TARGET_SIZE, batch_size=BATCH_SIZE, target_label=SHIRT_LABEL, processor=processor, model=model)

In [5]:
shirt_tensor.shape

torch.Size([461, 1024, 1024])

In [12]:
# Open the original video
cap = cv2.VideoCapture(input_video)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_idx = 0

# Prepare video writer
ret, frame = cap.read()
if not ret:
    raise RuntimeError("Cannot read video")

height, width = frame.shape[:2]
writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # reset to first frame

# Iterate over frames
while cap.isOpened() and frame_idx < len(shirt_tensor):
    ret, frame = cap.read()
    if not ret:
        break

    # Resize mask back to original frame size
    mask = shirt_tensor[frame_idx].numpy().astype(np.uint8) * 255
    mask_resized = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)

    # Convert mask to 3 channels
    mask_bgr = np.zeros_like(frame)        # shape (H, W, 3)
    mask_bgr[mask_resized > 0] = MASK_COLOR

    # Overlay mask
    blended = cv2.addWeighted(frame, 1 - ALPHA, mask_bgr, ALPHA, 0)

    writer.write(blended)
    frame_idx += 1

cap.release()
writer.release()
print(f"Output saved to {output_video}")

Output saved to data/new_output_2.mp4
